# Wan 2.2 I2V-A14B Video Generation with LoRAs

Animate a local starting image with the official Wan 2.2 image-to-video model. This notebook follows the image notebooks: install packages, edit settings, then run a cached pipeline that saves outputs and generation metadata by prompt.


This notebook uses `Models/Model_Families/WAN/` for its own assets.
Default checkpoint: the official base model (no local checkpoint is available). Default LoRA: `loras/Mouthful/05_wan22-mouthfull-140epoc-high-k3nk.safetensors`.
Run with the kernel working directory anywhere inside this project. Checkpoint and LoRA paths are relative to this family's `checkpoints/` and `loras/` folders; absolute paths also work. `.env`, relative `CACHE_LOCATION`, input images, and output folders resolve from the project root.


### Install Packages

Use the `huggingface` kernel with your Thor-compatible CUDA/PyTorch environment. This cell installs the inference and video-export dependencies; it does not explicitly install PyTorch, torchvision, xformers, or FlashAttention. After installation, `%pip check` checks the environment for dependency conflicts. Restart the kernel before running the pipeline.

MP4 export requires FFmpeg. `imageio-ffmpeg` can use a bundled executable or a system installation; on ARM64, install system `ffmpeg` if an executable is unavailable.

This shared environment uses Transformers 5 and Hugging Face Hub 1.32 or newer. Capping Transformers below 5 forces an older Hub version that conflicts with `datasets 5.1.0`. The bounds below let pip restore the compatible stack without downgrading `datasets`.


In [ ]:
# Keep the CUDA-enabled PyTorch build supplied for your JetPack/Thor environment.
from importlib.metadata import version
print(f"Installed PyTorch: {version('torch')}")

%pip install --upgrade "diffusers>=0.42,<1" "transformers>=5,<6" "accelerate>=1.2" "peft>=0.17" "huggingface_hub>=1.32,<3" safetensors python-dotenv sentencepiece ftfy pillow numpy imageio imageio-ffmpeg

# Check all installed packages, including datasets in the shared environment.
%pip check


## Run Pipeline

1. Set `input_image` to an existing local image (for example, a PNG saved by the SDXL or FLUX notebook).
2. Edit the motion prompts and settings below, then run the settings and pipeline cells. Each prompt animates the same starting image. Variations run sequentially.
3. Find MP4s, JSON metadata, and the exact resized/cropped starting frames in `Videos/<prompt>/`. `saved_paths` contains the MP4 paths from the latest run.

The defaults use the standard base model: 480p-area output, 81 frames, 16 fps, 40 steps, and guidance 3.5. With `preserve_aspect_ratio = True`, width/height set the pixel budget and the actual dimensions follow your image. Disable it for an exact center-cropped size. Raising resolution or frame count increases memory and generation time.

The first pipeline run downloads the complete Diffusers model, including **both** A14B experts. The text encoder and experts use BF16; the VAE uses FP32 with optional tiling. Components stay on CUDA, which shares Thor's system memory. Reruns reuse the loaded pipeline. `CACHE_LOCATION` (the cache root, with a `hub/` subdirectory) and optional `HUGGINGFACE_TOKEN` are read from `.env`, matching the other notebooks. A local checkpoint must be a complete Diffusers directory, not a single SDXL/ComfyUI weight file.

Optional LoRAs load separately into the high-noise and low-noise experts. Use matching Wan 2.2 **I2V-A14B** files supported by Diffusers. The default sampling settings are for the base model; acceleration/distillation LoRAs may require their own scheduler, guidance, and step settings.

References: [official model and Diffusers example](https://huggingface.co/Wan-AI/Wan2.2-I2V-A14B-Diffusers), [Wan pipeline API](https://huggingface.co/docs/diffusers/api/pipelines/wan), [Wan LoRA loader](https://huggingface.co/docs/diffusers/api/loaders/lora#diffusers.loaders.WanLoraLoaderMixin).


In [4]:
# Edit these settings, then run this cell and the pipeline cell below.
from pathlib import Path

# Find the project from either its root or a notebook's nested working directory.
project_root = next(
    (parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (parent / "Models" / "Model_Families").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start the notebook kernel inside the Image_Generator project.")
model_family = "WAN"
family_dir = project_root / "Models" / "Model_Families" / model_family
checkpoint_root = family_dir / "checkpoints"
lora_root = family_dir / "loras"

def resolve_path(value, relative_to):
    """Resolve relative paths against an explicit root; preserve absolute paths."""
    path = Path(value).expanduser()
    return (path if path.is_absolute() else relative_to / path).resolve()

# Every prompt animates this same starting image. Use an image from Images/ or any local path.
input_image = "Images/input.png"
prompts = [
    "The camera slowly moves forward. A gentle breeze moves through the scene, cinematic lighting, smooth natural motion as this beautiful woman's mouth moves up and down the cock",
]

# Diffusers loads both experts, the text encoder, VAE, tokenizer, and scheduler.
model_id = "Wan-AI/Wan2.2-I2V-A14B-Diffusers"
# Optional complete local Diffusers model directory under this family's checkpoints/,
# or a local path. None = download/reuse the official model in the Hugging Face cache.
# A single SDXL checkpoint or a ComfyUI transformer file is not a complete Wan pipeline.
checkpoint_dir = None

height = 480                  # Starting resolution budget; try 720 and 1280 after testing.
width = 832                   # Dimensions must be multiples of 16.
preserve_aspect_ratio = True  # Fit the source aspect ratio to height * width pixels.
# False = center-crop the image to the exact width/height above.
num_frames = 161               # Must be 4*n + 1; 161 frames at 16 fps is about 10 seconds.
fps = 16                      # MP4 playback rate; changing this does not generate more frames.
num_inference_steps = 40      # Standard base-model sampling, not a four-step distilled workflow.
guidance_scale = 3.5          # High-noise expert; > 1 enables negative prompting.
guidance_scale_2 = 3.5        # Low-noise expert; normally keep both guidance scales equal.
negative_prompt = "blurry, low quality, distorted, flickering, subtitles, watermark"
num_videos_per_prompt = 1     # Variations generated sequentially to limit peak memory.
seed = None                   # None = random; an integer = repeatable seed sequence.
enable_vae_tiling = True      # Reduce memory needed when encoding/decoding video.

# Optional Wan 2.2 I2V-A14B-compatible LoRAs. One local high-noise placeholder is selected.
# Use paths relative to this family's loras/ folder, or absolute paths. Set either expert's file
# to None to leave that expert unchanged. SDXL/FLUX/TI2V-5B LoRAs are incompatible.
# Rerun both cells after changing the model or LoRA settings.
Use_LoRA = True
LoRA_High_Noise_Weights = "Mouthful/05_wan22-mouthfull-140epoc-high-k3nk.safetensors"
LoRA_Low_Noise_Weights = None
LoRA_Scale = 0.8
# Add any trigger words required by your selected LoRA to the prompts.

# Relative input/output paths are anchored at the project root.
output_dir = "Videos"
show_videos = True            # Display saved MP4s inline.


In [5]:
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import re
import secrets
from time import perf_counter
from uuid import uuid4

from dotenv import load_dotenv
from IPython.display import Video, display
from PIL import Image, ImageOps
import torch
import diffusers
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video
import imageio_ffmpeg

# Validate settings and local inputs before downloading weights or releasing a model.
if not isinstance(prompts, (list, tuple)) or not prompts:
    raise ValueError("prompts must be a nonempty list of strings.")
if any(not isinstance(prompt, str) or not prompt.strip() for prompt in prompts):
    raise ValueError("Every prompt must be a nonempty string.")
unique_prompts = list(dict.fromkeys(prompts))
if not isinstance(model_id, str) or not model_id.strip():
    raise ValueError("model_id must be a nonempty Hugging Face model ID.")
for name, value in {"height": height, "width": width}.items():
    if type(value) is not int or value <= 0 or value % 16:
        raise ValueError(f"{name} must be a positive integer divisible by 16.")
for name, value in {
    "num_inference_steps": num_inference_steps,
    "num_videos_per_prompt": num_videos_per_prompt,
}.items():
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} must be a positive integer.")
if type(num_frames) is not int or num_frames < 5 or (num_frames - 1) % 4:
    raise ValueError("num_frames must be 4*n + 1 and at least 5 (for example, 49 or 81).")
if seed is not None and (type(seed) is not int or not 0 <= seed < 2**63):
    raise ValueError("seed must be None or an integer between 0 and 2**63 - 1.")
for name, value in {
    "guidance_scale": guidance_scale,
    "guidance_scale_2": guidance_scale_2,
    "LoRA_Scale": LoRA_Scale,
    "fps": fps,
}.items():
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value < 0:
        raise ValueError(f"{name} must be a finite number >= 0.")
if fps == 0:
    raise ValueError("fps must be greater than zero.")
# Diffusers decides whether to encode a negative prompt from the high-noise scale.
if (guidance_scale > 1) != (guidance_scale_2 > 1):
    raise ValueError("Both guidance scales must be > 1 for CFG, or both <= 1 to disable it.")
if not isinstance(negative_prompt, str):
    raise ValueError("negative_prompt must be a string; use an empty string to leave it blank.")
for name, value in {
    "Use_LoRA": Use_LoRA,
    "preserve_aspect_ratio": preserve_aspect_ratio,
    "enable_vae_tiling": enable_vae_tiling,
    "show_videos": show_videos,
}.items():
    if type(value) is not bool:
        raise ValueError(f"{name} must be True or False.")

if not isinstance(input_image, (str, Path)) or not str(input_image).strip():
    raise ValueError("Set input_image to a local starting image, for example Images/input.png.")
input_path = resolve_path(input_image, project_root)
if not input_path.is_file():
    raise FileNotFoundError(f"Starting image not found: {input_path}. Set input_image in the settings cell.")
input_sha256 = hashlib.sha256(input_path.read_bytes()).hexdigest()
with Image.open(input_path) as opened_image:
    source_image = ImageOps.exif_transpose(opened_image).convert("RGB")
source_size = source_image.size

checkpoint_path = None
checkpoint_key = None
if checkpoint_dir is not None:
    if not isinstance(checkpoint_dir, (str, Path)) or not str(checkpoint_dir).strip():
        raise ValueError("checkpoint_dir must be a local Diffusers directory, or None.")
    checkpoint_path = resolve_path(checkpoint_dir, checkpoint_root)
    if not checkpoint_path.is_dir() or not (checkpoint_path / "model_index.json").is_file():
        raise ValueError(f"Expected a complete Diffusers model directory with model_index.json: {checkpoint_path}")
    # Also reload if local config or weights are replaced at the same path.
    checkpoint_key = (
        str(checkpoint_path),
        tuple(
            (str(path.relative_to(checkpoint_path)), path.stat().st_size, path.stat().st_mtime_ns)
            for path in sorted(checkpoint_path.rglob("*"))
            if path.is_file() and path.suffix in {".json", ".safetensors"}
        ),
    )

lora_paths = {}
lora_key = None
if Use_LoRA:
    for expert, weights in {
        "high_noise": LoRA_High_Noise_Weights,
        "low_noise": LoRA_Low_Noise_Weights,
    }.items():
        if weights is None:
            continue
        if not isinstance(weights, (str, Path)) or not str(weights).strip():
            raise ValueError(f"Select a local {expert} LoRA .safetensors file, or None.")
        path = resolve_path(weights, lora_root)
        if not path.is_file():
            raise FileNotFoundError(f"LoRA file not found: {path}")
        if path.suffix.lower() != ".safetensors":
            raise ValueError("Select local .safetensors LoRA files.")
        lora_paths[expert] = path
    if not lora_paths:
        raise ValueError("Use_LoRA is True; select at least one high-noise or low-noise LoRA file.")
    lora_key = tuple(
        (expert, str(path), path.stat().st_size, path.stat().st_mtime_ns)
        for expert, path in lora_paths.items()
    )

# Check MP4 encoding support before loading the large model.
imageio_ffmpeg.get_ffmpeg_exe()
load_dotenv(project_root / ".env")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a kernel with the Thor-compatible CUDA build of PyTorch.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This notebook requires CUDA BF16 support for Wan 2.2 I2V-A14B.")

device = torch.device("cuda:0")
dtype = torch.bfloat16
gib = 1024**3
free_bytes, total_bytes = torch.cuda.mem_get_info(device)
print(f"GPU: {torch.cuda.get_device_name(device)} | CUDA: {torch.version.cuda}")
print(f"CUDA memory: {free_bytes / gib:.1f} GiB free / {total_bytes / gib:.1f} GiB total")

# CACHE_LOCATION is the HF_HOME root; retain its hub/ layout from the other notebooks.
cache_root = resolve_path(os.getenv("CACHE_LOCATION") or "Models", project_root)
cache_dir = cache_root / "hub"
cache_dir.mkdir(parents=True, exist_ok=True)
model_source = str(checkpoint_path) if checkpoint_path else model_id
load_options = {
    "cache_dir": str(cache_dir),
    "token": os.getenv("HUGGINGFACE_TOKEN") or None,
    "local_files_only": checkpoint_path is not None,
    "use_safetensors": True,
    "low_cpu_mem_usage": True,
}

# Keep components on CUDA: CPU offloading also consumes Thor's shared system RAM.
# Reuse the pipeline when only prompts, source image, or generation settings change.
pipeline_config = (model_source, checkpoint_key, str(cache_dir.resolve()), dtype, str(device))
if (
    "pipe" not in globals()
    or not isinstance(pipe, WanImageToVideoPipeline)
    or globals().get("_loaded_wan_config") != pipeline_config
):
    if "pipe" in globals():
        del pipe
    gc.collect()
    torch.cuda.empty_cache()
    # FP32 VAE follows Diffusers guidance for stable Wan encoding/decoding.
    pipe = WanImageToVideoPipeline.from_pretrained(
        model_source,
        vae=AutoencoderKLWan.from_pretrained(
            model_source, subfolder="vae", torch_dtype=torch.float32, **load_options
        ),
        torch_dtype=dtype,
        **load_options,
    ).to(device)
    _loaded_wan_config = pipeline_config
else:
    print("Reusing the loaded pipeline.")
if pipe.transformer is None or pipe.transformer_2 is None or pipe.config.boundary_ratio is None:
    raise ValueError("Select a Wan 2.2 I2V-A14B Diffusers pipeline with both high-noise and low-noise experts.")
if enable_vae_tiling:
    pipe.vae.enable_tiling()
else:
    pipe.vae.disable_tiling()
print(f"Model: {model_source} | Expert boundary: {pipe.config.boundary_ratio}")

# Load separate LoRAs into the matching experts; never fuse into the base weights.
if getattr(pipe, "_notebook_lora_key", None) != lora_key:
    pipe.unload_lora_weights()
    pipe._notebook_lora_key = None
    try:
        for expert, path in lora_paths.items():
            pipe.load_lora_weights(
                str(path.parent),
                weight_name=path.name,
                adapter_name="notebook_lora",
                load_into_transformer_2=(expert == "low_noise"),
                local_files_only=True,
            )
    except Exception as exc:
        pipe.unload_lora_weights()
        raise RuntimeError("Could not load the LoRAs. Check Wan 2.2 I2V-A14B and expert compatibility.") from exc
    pipe._notebook_lora_key = lora_key
if Use_LoRA:
    pipe.set_adapters("notebook_lora", adapter_weights=float(LoRA_Scale))
    print(f"LoRAs: {lora_paths} | Strength: {LoRA_Scale}")
else:
    print("LoRA disabled; using the base model.")
lora_settings = {
    "enabled": Use_LoRA,
    "high_noise_weights": str(lora_paths["high_noise"]) if "high_noise" in lora_paths else None,
    "low_noise_weights": str(lora_paths["low_noise"]) if "low_noise" in lora_paths else None,
    "scale": float(LoRA_Scale) if Use_LoRA else 0.0,
}

for name in ("transformer", "transformer_2", "text_encoder", "vae"):
    component = getattr(pipe, name)
    print(f"{name}: {component.device}, {component.dtype}")
del component

# Match the source aspect ratio to the requested area and Wan's spatial patch size.
height_multiple = pipe.vae_scale_factor_spatial * pipe.transformer.config.patch_size[1]
width_multiple = pipe.vae_scale_factor_spatial * pipe.transformer.config.patch_size[2]
if preserve_aspect_ratio:
    aspect_ratio = source_image.height / source_image.width
    video_height = max(height_multiple, int(math.sqrt(height * width * aspect_ratio)) // height_multiple * height_multiple)
    video_width = max(width_multiple, int(math.sqrt(height * width / aspect_ratio)) // width_multiple * width_multiple)
    conditioning_image = source_image.resize((video_width, video_height), Image.Resampling.LANCZOS)
else:
    video_height, video_width = height, width
    if video_height % height_multiple or video_width % width_multiple:
        raise ValueError(f"This model requires height/width multiples of {height_multiple}/{width_multiple}.")
    conditioning_image = ImageOps.fit(source_image, (video_width, video_height), method=Image.Resampling.LANCZOS)
print(f"Input: {input_path.name} {source_size} -> {video_width}x{video_height}")
print(f"Video: {num_frames} frames at {fps} fps ({num_frames / fps:.2f}s)")

output_root = resolve_path(output_dir, project_root)
output_root.mkdir(parents=True, exist_ok=True)
saved_paths = []
torch.cuda.reset_peak_memory_stats(device)

# Process prompts and variations sequentially; never batch several A14B videos in RAM.
with torch.inference_mode():
    for prompt_index, prompt in enumerate(unique_prompts):
        slug = re.sub(r"[^a-z0-9]+", "-", prompt.lower()).strip("-")[:60].rstrip("-") or "prompt"
        prompt_hash = hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:16]
        prompt_dir = output_root / f"{slug}-{prompt_hash}"
        prompt_dir.mkdir(parents=True, exist_ok=True)
        (prompt_dir / "prompt.txt").write_text(prompt, encoding="utf-8")

        for video_index in range(num_videos_per_prompt):
            video_seed = (
                secrets.randbelow(2**63) if seed is None
                else (seed + prompt_index * num_videos_per_prompt + video_index) % (2**63)
            )
            generator = torch.Generator(device=device).manual_seed(video_seed)
            generation_settings = {
                "height": video_height,
                "width": video_width,
                "num_frames": num_frames,
                "num_inference_steps": num_inference_steps,
                "guidance_scale": guidance_scale,
                "guidance_scale_2": guidance_scale_2,
                "negative_prompt": negative_prompt,
                "num_videos_per_prompt": 1,
                "output_type": "np",
            }
            print(f"Prompt {prompt_index + 1}/{len(unique_prompts)}, video {video_index + 1}/{num_videos_per_prompt}: {prompt}")
            torch.cuda.synchronize(device)
            start = perf_counter()
            frames = pipe(
                image=conditioning_image, prompt=prompt, generator=generator, **generation_settings
            ).frames[0]
            torch.cuda.synchronize(device)
            elapsed = perf_counter() - start

            # Unique filenames preserve previous runs, including repeated seeds.
            output_path = prompt_dir / f"wan22-i2v-seed-{video_seed}-{uuid4().hex}.mp4"
            export_to_video(frames, str(output_path), fps=fps, macro_block_size=1)
            frame_count = len(frames)
            del frames  # Do not retain a previous decoded video during the next generation.
            conditioning_path = output_path.with_suffix(".input.png")
            conditioning_image.save(conditioning_path)
            metadata = {
                "prompt": prompt,
                "model_id": model_id,
                "checkpoint_dir": str(checkpoint_path) if checkpoint_path else None,
                "input_image": str(input_path),
                "input_sha256": input_sha256,
                "input_size": list(source_size),
                "conditioning_image": conditioning_path.name,
                "preserve_aspect_ratio": preserve_aspect_ratio,
                "requested_height": height,
                "requested_width": width,
                "lora": dict(lora_settings),
                "seed": video_seed,
                "dtype": str(dtype),
                "vae_dtype": str(pipe.vae.dtype),
                "enable_vae_tiling": enable_vae_tiling,
                "device": str(device),
                "torch_version": torch.__version__,
                "diffusers_version": diffusers.__version__,
                "scheduler": dict(pipe.scheduler.config),
                "boundary_ratio": pipe.config.boundary_ratio,
                "fps": fps,
                "duration_seconds": frame_count / fps,
                "generation_seconds": elapsed,
                "videos_requested_per_prompt": num_videos_per_prompt,
                **generation_settings,
            }
            output_path.with_suffix(".json").write_text(
                json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
            )
            saved_paths.append(output_path)
            print(f"Saved {output_path} | Generation: {elapsed:.1f}s")
            if show_videos:
                display(Video(filename=str(output_path), embed=True, html_attributes="controls loop"))

print(f"Saved {len(saved_paths)} videos across {len(unique_prompts)} prompt folders.")
print(f"Peak PyTorch CUDA allocation: {torch.cuda.max_memory_allocated(device) / gib:.1f} GiB")


GPU: NVIDIA Thor | CUDA: 13.2
CUDA memory: 6.7 GiB free / 122.9 GiB total
Reusing the loaded pipeline.
Model: Wan-AI/Wan2.2-I2V-A14B-Diffusers | Expert boundary: 0.9
LoRAs: {'high_noise': PosixPath('/home/malneyugnfl/SoftwareDev/Image_Generator/Models/Model_Families/WAN/loras/Mouthful/05_wan22-mouthfull-140epoc-high-k3nk.safetensors')} | Strength: 0.8
transformer: cuda:0, torch.bfloat16
transformer_2: cuda:0, torch.bfloat16
text_encoder: cuda:0, torch.bfloat16
vae: cuda:0, torch.float32
Input: input.png (1024, 1024) -> 624x624
Video: 161 frames at 16 fps (10.06s)
Prompt 1/1, video 1/1: The camera slowly moves forward. A gentle breeze moves through the scene, cinematic lighting, smooth natural motion as this beautiful woman's mouth moves up and down the cock


  0%|          | 0/40 [00:00<?, ?it/s]

Saved /home/malneyugnfl/SoftwareDev/Image_Generator/Videos/the-camera-slowly-moves-forward-a-gentle-breeze-moves-throug-291ecadbf3f14c7f/wan22-i2v-seed-7620473049280966237-f737c4058ca34aa1a25e3ab5338e4512.mp4 | Generation: 9414.7s


Saved 1 videos across 1 prompt folders.
Peak PyTorch CUDA allocation: 72.6 GiB
